<a href="https://colab.research.google.com/github/hyui02/ESAA_assignment/blob/main/ESAA_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 수상작 리뷰: Mercari Price Suggestion Challenge (1위 팀 핵심 모델)

- 리뷰 노트북: [Mercari Golf: 0.3875 CV in 75 LOC, 1900 s](https://www.kaggle.com/code/lopuhin/mercari-golf-0-3875-cv-in-75-loc-1900-s) (1위 팀 Konstantin Lopukhin의 축약 버전)
- 대회: [Mercari Price Suggestion Challenge (Kaggle, 2017~2018)](https://www.kaggle.com/competitions/mercari-price-suggestion-challenge)
- 한 줄 요약: 상품명과 설명 텍스트를 TF-IDF로 바꾼 20만 차원 희소 행렬을 그대로 작은 MLP(192-64-64-1)에 넣고, 입력 형태만 다르게 한 MLP 4개를 CPU 코어 하나씩 맡겨 동시에 학습한 뒤 평균하여 75줄, 약 32분 만에 검증 RMSLE 0.387을 얻었다.

---

# **1. 데이터 설명**

## 1-1. 대회 과제
- 일본 중고거래 앱 **메르카리(Mercari)** 에 판매자가 입력한 상품 정보(상품명, 카테고리, 브랜드, 상품 상태, 설명 등)로 **판매 가격(USD)을 예측**하는 회귀 문제이다.
- 판매자가 상품을 올릴 때 적절한 가격을 자동으로 추천해 주는 기능을 만드는 것이 목적이다.
- 평가지표는 **RMSLE(Root Mean Squared Logarithmic Error)** 이다. 예측값과 실제값에 각각 1을 더해 로그를 취한 뒤 RMSE를 구한다.
    - 가격이 비쌀수록 오차가 커지는 영향을 줄이고, 비싼 상품과 싼 상품의 **상대적인 오차**를 비슷한 비중으로 평가한다.
    - 그래서 타깃을 `log1p(price)`로 바꿔 MSE로 학습하면 평가지표를 그대로 최적화하는 셈이 된다.
- 제출 파일은 `test_id`, `price` 두 열로 구성한다.

## 1-2. 대회 진행 방식
- 데이터가 공개된 실제 거래 정보이기 때문에 **커널 전용(Kernels Only) 대회**로 진행되었다. 제출은 Kaggle 커널에서만 가능하다.
- 대회는 두 단계로 진행되었다.
    - **1단계(2017.11.22 ~ 2018.02.14)**: 테스트 데이터 약 70만 행으로 공개 리더보드 점수를 확인한다.
    - **2단계(2018.02.15 ~ 02.21)**: 참가자가 고른 커널을 **약 350만 행의 새 테스트 데이터**로 다시 실행해 최종 순위를 정한다. 이 단계에서는 코드를 고칠 수 없다.
- **커널 자원 제한**: CPU 4코어, 메모리 16GB, 실행 시간 60분, 디스크 1GB이다. 학습과 예측, 앙상블까지 **하나의 커널 안에서** 끝내야 한다.
    - 따라서 이 대회는 성능만큼이나 **제한된 시간과 메모리 안에서 얼마나 효율적으로 학습하는지**가 중요했다.
- 상금은 1~3위에 각각 6만, 3만, 1만 달러였고 2,380개 팀이 참가하였다.

## 1-3. 파일 구성
| 파일 | 내용 |
|---|---|
| `train.tsv` | 학습 데이터. 상품 정보와 가격(`price`)이 들어 있다. 탭으로 구분된 파일이다. |
| `test.tsv` | 테스트 데이터. `price`가 없다. 2단계에서 약 350만 행짜리 파일로 바뀐다. |
| `sample_submission.csv` | 제출 파일 형식 예시 |

## 1-4. 컬럼과 노트북의 사용 여부
| 컬럼 | 내용 | 사용 |
|---|---|---|
| `train_id` / `test_id` | 상품 ID | × |
| `name` | 상품명. 가격처럼 보이는 텍스트(예: `$20`)는 누수 방지를 위해 `[rm]`으로 바뀌어 있다. | ○ |
| `item_condition_id` | 판매자가 입력한 상품 상태 | ○ |
| `category_name` | 카테고리 | ○ (텍스트에 합침) |
| `brand_name` | 브랜드명 | ○ (상품명에 합침) |
| `price` | **타깃.** 판매 가격(USD) | ○ |
| `shipping` | 배송비 부담 주체. 1이면 판매자, 0이면 구매자 | ○ |
| `item_description` | 상품 설명 전문. `name`과 마찬가지로 가격 텍스트는 `[rm]`으로 바뀌어 있다. | ○ (텍스트에 합침) |

- 이 노트북은 ID를 뺀 **모든 컬럼을 사용**하지만, 카테고리·브랜드를 따로 인코딩하지 않고 **텍스트에 이어 붙여** 한꺼번에 처리한다.

## 1-5. 데이터를 다룰 때 알아 둘 특징
- **텍스트 비중이 크다**: 같은 카테고리라도 상품명과 설명에 들어간 단어(브랜드, 모델명, 용량, 새상품 여부 등)에 따라 가격 차이가 크다. 그래서 텍스트를 얼마나 잘 표현하느냐가 성능을 좌우한다.
- **가격이 0인 행**: 학습 데이터에 가격이 0인 상품이 있으며, 노트북은 이를 학습에서 뺀다(`train['price'] > 0`).
- **가격 분포가 한쪽으로 치우쳐 있다**: 대부분이 저가 상품이고 고가 상품이 길게 꼬리를 이룬다. 로그 변환이 필요한 이유이다.
- **2단계 데이터 크기**: 테스트 데이터가 5배로 늘어나므로, 1단계에서 시간이 빠듯한 코드는 2단계에서 실패할 위험이 있었다.

---

# **2. 코드 흐름**

1. **환경 설정**
    - `OMP_NUM_THREADS=1`로 라이브러리가 쓰는 스레드를 1개로 제한하고, 실행 시간을 재는 `timer` 함수를 만든다.
2. **데이터 불러오기와 검증 세트 분리**
    - `train.tsv`를 읽고 가격이 0인 행을 뺀다.
    - `KFold(n_splits=20)`의 **첫 번째 분할만** 사용하여 학습 95%, 검증 5%로 나눈다.
3. **전처리(`preprocess`)**
    - `name`에 `brand_name`을 붙이고, `item_description` + `name` + `category_name`을 합쳐 `text` 컬럼을 만든다.
4. **타깃 변환**
    - `log1p(price)`를 취한 뒤 `StandardScaler`로 평균 0, 표준편차 1이 되게 맞춘다.
5. **벡터화(`make_union`)**
    - `name`: 단어 단위 TF-IDF(최대 10만 개 단어)
    - `text`: 단어 1~2-gram TF-IDF(최대 10만 개)
    - `shipping`, `item_condition_id`: `DictVectorizer`
    - 세 결과를 옆으로 이어 붙여 **약 140만 행 × 20만 열의 희소 행렬**을 만든다.
6. **입력 두 종류 준비**
    - TF-IDF 값 그대로인 행렬과, 0이 아닌 값을 모두 1로 바꾼 **이진화(binary) 행렬**을 만든다.
7. **희소 MLP 학습(`fit_predict`)**
    - 희소 입력 → Dense 192 → 64 → 64 → 1 구조이다.
    - Adam(학습률 3e-3)으로 3에폭 학습하며, 에폭마다 **배치 크기를 2배**로 키운다(2048 → 4096 → 8192).
8. **앙상블**
    - 이진화 입력 2개 + TF-IDF 입력 2개, 총 4개 모델을 `ThreadPool`로 **동시에** 학습하고 예측을 평균한다.
9. **평가**
    - 스케일링과 로그 변환을 되돌려 원래 가격으로 바꾼 뒤 검증 RMSLE를 출력한다(실행 로그 기준 **0.3872**, 총 1,935초).

---

# **3. 주요 코드 분석**

- 아래 코드 셀은 원본 노트북의 핵심 로직을 그대로 옮기되, 대회 데이터 없이도 에러 없이 실행되도록 **작은 예시 데이터**를 넣은 것이다.
- 원본은 2017년 당시의 TensorFlow 1 문법(`tf.Session`, `tf.ConfigProto`)으로 작성되어 지금의 Colab에서는 그대로 돌아가지 않는다. 모델 부분은 같은 구조를 **현재 버전의 Keras 문법**으로 바꿔 적었다.
- 첫 코드 셀(라이브러리 불러오기)을 먼저 실행한 뒤 위에서부터 차례로 실행한다.

In [ ]:
import os; os.environ['OMP_NUM_THREADS'] = '1'
from contextlib import contextmanager
from functools import partial
from operator import itemgetter
from multiprocessing.pool import ThreadPool
import time
from typing import List, Dict

import numpy as np
import pandas as pd
import tensorflow as tf
import keras as ks
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer as Tfidf
from sklearn.pipeline import make_pipeline, make_union, Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import KFold

tf.get_logger().setLevel('ERROR')   # 반복 학습 시 나오는 TensorFlow 경고 숨김

**① 실행 시간 측정용 `timer`**

In [ ]:
@contextmanager
def timer(name):
    t0 = time.time()
    yield
    print(f'[{name}] done in {time.time() - t0:.2f} s')

with timer('예시 작업'):
    s = sum(range(10_000_000))

- `@contextmanager`로 만든 함수는 `with` 문과 함께 쓸 수 있다. `yield` 앞은 블록에 들어갈 때, 뒤는 블록을 빠져나올 때 실행된다.
- 원본은 데이터 처리, 각 에폭, 모델 학습 전체에 이 함수를 감싸 두었다. 실행 시간 60분 제한이 있는 대회였기 때문에 **어느 단계가 시간을 얼마나 쓰는지**를 계속 확인한 것이다.
- 실제 로그를 보면 학습 데이터 처리 217초, 1에폭 약 925초, 2에폭 약 495초, 3에폭 약 270초가 걸렸다.

**② 텍스트 필드 합치기(`preprocess`)**

In [ ]:
# 예시 데이터: 상품 5개 (실제 컬럼 구성과 같음)
train = pd.DataFrame({
    'name': ['MLB Cincinnati Reds T Shirt Size XL', 'Razer BlackWidow Chroma Keyboard',
             'AVA-VIV Blouse', 'Leather Horse Statues', '24K GOLD plated rose'],
    'item_condition_id': [3, 3, 1, 1, 1],
    'category_name': ['Men/Tops/T-shirts', 'Electronics/Computers & Tablets/Components & Parts',
                      'Women/Tops & Blouses/Blouse', 'Home/Home Décor/Home Décor Accents',
                      'Women/Jewelry/Necklaces'],
    'brand_name': [np.nan, 'Razer', 'Target', np.nan, np.nan],
    'price': [10.0, 52.0, 10.0, 35.0, 44.0],
    'shipping': [1, 0, 1, 1, 0],
    'item_description': ['No description yet',
                         'This keyboard is in great condition and works like it came out of the box.',
                         'Adorable top with a hint of lace and a key hole in the back!',
                         'New with tags. Leather horses. Retail for [rm] each.',
                         'Complete with certificate of authenticity'],
})

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df['name'] = df['name'].fillna('') + ' ' + df['brand_name'].fillna('')
    df['text'] = (df['item_description'].fillna('') + ' ' + df['name'] + ' ' + df['category_name'].fillna(''))
    return df[['name', 'text', 'shipping', 'item_condition_id']]

prep = preprocess(train.copy())
prep

- 컬럼별로 따로 모델링하지 않고, **짧은 텍스트(`name` + 브랜드)** 와 **긴 텍스트(설명 + 상품명 + 카테고리)** 두 덩어리로 합쳤다.
- 브랜드와 카테고리를 별도의 범주형 변수로 원핫 인코딩하는 대신 텍스트 안의 단어로 넣었다. 이렇게 하면 `Razer`처럼 브랜드명에도, 상품명에도 나오는 단어를 같은 특징으로 묶을 수 있고, 결측(`NaN`)은 빈 문자열로 처리되어 따로 신경 쓸 필요가 없다.
- 상품명은 `name`과 `text` 양쪽에 들어가므로, 가격에 큰 영향을 주는 상품명 단어가 두 번 반영된다.

**③ 컬럼별 벡터화를 하나로 묶기(`on_field` + `make_union`)**

In [ ]:
def on_field(f: str, *vec) -> Pipeline:
    return make_pipeline(FunctionTransformer(itemgetter(f), validate=False), *vec)

def to_records(df: pd.DataFrame) -> List[Dict]:
    return df.to_dict(orient='records')

vectorizer = make_union(
    on_field('name', Tfidf(max_features=100000, token_pattern=r'\w+')),
    on_field('text', Tfidf(max_features=100000, token_pattern=r'\w+', ngram_range=(1, 2))),
    on_field(['shipping', 'item_condition_id'],
             FunctionTransformer(to_records, validate=False), DictVectorizer()),
    n_jobs=1)   # 원본은 n_jobs=4

X_train = vectorizer.fit_transform(prep).astype(np.float32)
print('X_train:', X_train.shape, 'of', X_train.dtype, '/ 희소 행렬 형식:', type(X_train).__name__)

n_name = len(vectorizer.transformer_list[0][1][-1].vocabulary_)
n_text = len(vectorizer.transformer_list[1][1][-1].vocabulary_)
print('name 단어 수:', n_name, '/ text 단어·2-gram 수:', n_text)
print('DictVectorizer가 만든 열:', vectorizer.transformer_list[2][1][-1].feature_names_)

- `on_field('name', ...)`는 **"DataFrame에서 `name` 열만 꺼낸 뒤 TF-IDF를 적용하라"** 는 파이프라인이다. `itemgetter(f)`가 `df[f]`와 같은 역할을 한다.
- `make_union`은 여러 파이프라인의 결과를 **옆으로 이어 붙인다**. 세 가지 벡터화가 한 번의 `fit_transform`으로 끝나고, 검증·테스트 데이터에도 `transform` 한 줄로 같은 처리를 할 수 있다.
- `token_pattern=r'\w+'`로 한 글자 단어와 숫자도 토큰으로 남겼다. 사이즈(`XL`), 용량(`64`), 수량 같은 정보가 가격과 관련이 크기 때문이다. 기본값은 두 글자 이상만 남긴다.
- `text`에는 1~2-gram을 써서 `great condition`, `brand new`처럼 **두 단어 조합**의 의미도 잡는다.
- `DictVectorizer`는 숫자 값을 원핫 인코딩하지 않고 **그대로 한 열**로 둔다. 그래서 `shipping`, `item_condition_id`는 2개 열이 되고, 실제 로그의 입력 크기도 10만 + 10만 + 2 = **200,002열**이다.

**④ 타깃 변환과 검증 세트 분리**

In [ ]:
# 예시: 가격이 0인 상품 2개가 섞인 102개 데이터
rng = np.random.default_rng(0)
df = pd.DataFrame({'price': np.r_[0, 0, rng.lognormal(3, 0.8, 100).round(0) + 1]})

df = df[df['price'] > 0].reset_index(drop=True)                    # 가격 0 제거
cv = KFold(n_splits=20, shuffle=True, random_state=42)
train_ids, valid_ids = next(cv.split(df))                          # 첫 번째 분할만 사용
tr, va = df.iloc[train_ids], df.iloc[valid_ids]
print('학습:', len(tr), '행 / 검증:', len(va), '행')

y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(np.log1p(tr['price'].values.reshape(-1, 1)))
print('변환 후 평균·표준편차:', y_train.mean().round(3), y_train.std().round(3))

# 예측값을 원래 가격으로 되돌리는 순서: 스케일 복원 → expm1
y_back = np.expm1(y_scaler.inverse_transform(y_train)[:, 0])
print('복원 확인:', np.allclose(y_back, tr['price'].values))

- 타깃은 `log1p` → `StandardScaler` 순서로 바꾸고, 예측 후에는 반대 순서(`inverse_transform` → `expm1`)로 되돌린다.
- 로그 변환된 값에 MSE를 쓰면 평가지표인 RMSLE를 직접 줄이는 것과 같다. `StandardScaler`로 평균 0, 표준편차 1에 맞추면 마지막 출력층이 0 근처의 값만 내면 되어 학습이 안정된다.
- `KFold(n_splits=20)`에서 `next()`로 **첫 번째 분할 하나만** 꺼내 학습 95%, 검증 5%로 쓴다. 5%라도 약 7만 4천 행이라 검증용으로 충분히 크고, 20개 폴드를 모두 돌리면 시간 제한을 넘기기 때문이다.
- 다만 제목의 "CV"는 실제로는 **한 번의 홀드아웃 검증** 결과이다.

**⑤ 희소 입력 MLP (핵심)**

In [ ]:
# 예시 데이터: 6,000행 × 1,000열, 행마다 0이 아닌 값이 20개 정도인 희소 행렬
# 타깃은 "단어가 있는지"와 "단어의 가중치" 양쪽에 영향을 받도록 만들었다
import scipy.sparse as sp
X_demo = sp.random(6000, 1000, density=0.02, format='csr', dtype=np.float32, random_state=0)
true_w = np.random.default_rng(0).normal(size=1000).astype(np.float32)
noise = 0.5 * np.random.default_rng(1).normal(size=6000)
y_raw = X_demo.astype(bool).astype(np.float32) @ true_w + X_demo @ (2 * true_w) + noise
y_demo = StandardScaler().fit_transform(y_raw.reshape(-1, 1))
X_tr, X_va, y_tr, y_va = X_demo[:5000], X_demo[5000:], y_demo[:5000], y_demo[5000:]

def fit_predict(xs, y_train) -> np.ndarray:
    X_train, X_test = xs
    with timer('fit_predict'):
        model_in = ks.Input(shape=(X_train.shape[1],), dtype='float32', sparse=True)
        out = ks.layers.Dense(192, activation='relu')(model_in)
        out = ks.layers.Dense(64, activation='relu')(out)
        out = ks.layers.Dense(64, activation='relu')(out)
        out = ks.layers.Dense(1)(out)
        model = ks.Model(model_in, out)
        model.compile(loss='mean_squared_error', optimizer=ks.optimizers.Adam(learning_rate=3e-3))
        for i in range(3):
            with timer(f'epoch {i + 1} (batch_size={2**(5 + i)})'):   # 원본은 2**(11 + i)
                model.fit(x=X_train, y=y_train, batch_size=2**(5 + i), epochs=1, verbose=0)
        return model.predict(X_test, verbose=0)[:, 0]

pred = fit_predict([X_tr, X_va], y_tr)
print('검증 RMSE(스케일된 값 기준):', np.sqrt(np.mean((pred - y_va[:, 0]) ** 2)).round(4))

- **희소 입력을 그대로 받는다**: `ks.Input(..., sparse=True)`로 입력층을 만들면 20만 차원짜리 TF-IDF 행렬을 밀집 행렬로 바꾸지 않고 바로 넣을 수 있다. 140만 행 × 20만 열을 밀집 행렬로 만들면 메모리 16GB를 훨씬 넘지만, 희소 행렬은 0이 아닌 값만 저장하므로 가능하다.
- **첫 층이 사실상 임베딩 역할**: 희소 벡터와 첫 Dense 층(20만 × 192)의 곱은, 문장에 나온 단어들의 가중치 벡터를 TF-IDF 값으로 가중합한 것과 같다. 단어 임베딩을 따로 만들지 않고도 첫 층이 그 역할을 학습한다.
- **작고 단순한 구조**: 192 → 64 → 64 → 1의 얕은 MLP이다. RNN이나 CNN보다 훨씬 빨라서 제한 시간 안에 여러 개를 학습해 앙상블할 수 있었다. 1위 팀 발표 자료에서도 MLP를 고른 이유로 빠른 학습 속도, 텍스트와 범주형 특징 사이의 상호작용 포착, 모델 간 편차가 커서 앙상블 효과가 크다는 점을 들었다.
- **에폭마다 배치 크기를 2배로**: 2048 → 4096 → 8192로 키운다. 배치가 커지면 기울기의 잡음이 줄어 학습률을 낮추는 것과 비슷한 효과가 나고, 한 에폭에 걸리는 시간도 줄어든다. 실제로 에폭당 시간이 약 925초 → 495초 → 270초로 줄었다.
- **에폭 3번**: 데이터가 크고 모델이 빨리 과적합되므로 짧게 학습한다.
- 위 예시는 데이터가 작아서 배치 크기를 32 → 64 → 128로 줄였다. 원본의 `tf.Session`, `tf.ConfigProto` 부분은 TensorFlow 2에서 없어진 문법이라 빼고, 학습률 인자 이름도 `lr`에서 `learning_rate`로 바꿨다.

**⑥ 입력 두 종류 × 2번 학습, 4개 모델 동시 학습 앙상블**

In [ ]:
# 이진화: 0이 아닌 값을 모두 1로 (원본의 np.bool은 최신 넘파이에서 없어져 bool로 바꿈)
Xb_tr, Xb_va = [x.astype(bool).astype(np.float32) for x in [X_tr, X_va]]
print('TF-IDF 값 예시 :', X_tr[0].data[:5].round(3))
print('이진화 값 예시 :', Xb_tr[0].data[:5])

xs = [[Xb_tr, Xb_va], [X_tr, X_va]] * 2          # 이진 2개 + TF-IDF 2개 = 4개 모델
with ThreadPool(processes=4) as pool:
    preds = pool.map(partial(fit_predict, y_train=y_tr), xs)

for i, p in enumerate(preds):
    print(f'모델 {i + 1} ({"이진" if i % 2 == 0 else "TF-IDF"}) RMSE:', np.sqrt(np.mean((p - y_va[:, 0]) ** 2)).round(4))
y_pred = np.mean(preds, axis=0)
print('4개 평균 RMSE           :', np.sqrt(np.mean((y_pred - y_va[:, 0]) ** 2)).round(4))

- **같은 모델, 다른 입력**: 단어가 "얼마나 중요한가"(TF-IDF 값)를 보는 모델과 단어가 "있는가 없는가"(0/1)만 보는 모델을 함께 쓴다. 구조는 같아도 보는 정보가 달라서 예측 오차의 방향이 달라지고, 평균을 내면 오차가 서로 상쇄된다.
- 같은 입력으로도 2번씩 학습한다. 가중치 초기값이 무작위라 학습할 때마다 결과가 조금씩 달라지므로, 이것만으로도 앙상블 효과가 있다.
- **코어 하나에 모델 하나**: `OMP_NUM_THREADS=1`과 TensorFlow 스레드 1개 설정으로 모델 하나가 CPU 코어 하나만 쓰게 한 뒤, `ThreadPool(processes=4)`로 4개 모델을 동시에 학습한다. 한 모델이 4코어를 나눠 쓸 때 생기는 스레드 간 조율 비용을 피하는 방식이다. 로그를 보면 4개 모델이 각각 약 1,700초로 **나란히 끝나**, 차례로 학습했다면 걸렸을 시간의 약 4분의 1로 줄었다.
- 위 예시에서도 개별 모델보다 4개 평균의 RMSE가 더 낮게 나온다(학습할 때마다 값은 조금씩 달라진다).
- 평균한 예측은 ④의 역변환을 거쳐 원래 가격으로 되돌린 뒤 RMSLE를 계산한다. 실행 로그의 검증 RMSLE는 **0.3872**이다.
- 실제 1위 솔루션은 이 구조를 확장해 **데이터셋 3종 × 모델 4개**를 앙상블하였다. 이 노트북은 그중 핵심만 75줄로 줄여 보여 준 것이다.

---

# **4. 새로 알게 된 점 / 배운 점**

**배운 점**
- **고차원 희소 입력 + 단순한 MLP의 힘**: 텍스트 데이터라고 해서 꼭 RNN이나 사전학습 임베딩이 필요한 것은 아니었다. TF-IDF 20만 차원을 희소 행렬 그대로 작은 MLP에 넣는 것만으로 1위권 성능이 나왔다. 데이터와 제약 조건(시간·메모리)에 맞는 모델을 고르는 것이 중요하다는 점을 배웠다.
- **평가지표에 맞춘 타깃 변환**: RMSLE로 평가하는 대회이므로 `log1p` 변환 후 MSE로 학습하면 평가지표를 그대로 최적화하게 된다. 지난 회귀 실습(자전거 대여, 주택 가격)에서 배운 로그 변환이 실제 수상작에서도 똑같이 쓰였다.
- **입력을 바꿔서 앙상블 다양성 만들기**: 모델 종류를 바꾸지 않고 입력만 TF-IDF와 이진화로 다르게 해도 앙상블 효과가 생긴다는 점이 새로웠다.
- **배치 크기 키우기를 학습률 스케줄처럼 사용**: 에폭마다 배치를 2배로 키워 학습 후반에는 더 안정적으로, 더 빠르게 학습한다.
- **자원 제약을 설계에 반영**: 코어당 모델 하나를 배정해 4개 모델을 동시에 학습시킨 방식은, 모델 성능뿐 아니라 **주어진 자원을 어떻게 나눠 쓸지**도 설계의 일부라는 점을 보여 준다.
- **scikit-learn 파이프라인으로 짧고 재사용 가능한 전처리**: `make_union`과 `on_field`로 컬럼별 벡터화를 한 객체에 묶어, 학습·검증 데이터에 같은 처리를 한 줄로 적용하였다.

**아쉬운 점과 개선 방향**
- **제출 파일을 만들지 않는다**: 검증 RMSLE만 출력하고 `test.tsv` 예측과 제출 파일 저장 코드는 없다(출력 파일 0B). 실제 대회 제출용이 아니라 핵심 아이디어를 보여 주는 데모이다.
- **한 번의 홀드아웃 검증**: 제목에는 CV라고 되어 있지만 20개 폴드 중 첫 번째 분할 하나만 썼다. 검증 데이터가 크긴 하지만 분할에 따른 점수 편차는 알 수 없다.
- **`item_condition_id`를 숫자로 취급**: `DictVectorizer`는 숫자 값을 그대로 한 열로 두므로, 상품 상태 1~5가 크기 순서를 가진 연속형 값처럼 들어간다. 상태와 가격의 관계가 꼭 일직선이라는 보장이 없으므로, 문자열로 바꿔 원핫 인코딩하는 방법도 비교해 볼 만하다.
- **카테고리 구조를 활용하지 않음**: `Women/Tops & Blouses/Blouse`처럼 계층 구조가 있는 카테고리를 텍스트에 섞기만 했다. 대분류·중분류를 따로 변수로 만들면 추가 정보가 될 수 있다.
- **재현성 문제**: 2017년 당시 문법(`tf.Session`, `np.bool`, `lr` 인자)이라 지금 환경에서는 그대로 실행되지 않는다. 무작위 시드도 고정하지 않아 실행할 때마다 점수가 조금씩 달라진다.

**내 프로젝트에 적용할 점**
- 텍스트가 섞인 표 데이터를 다룰 때, 복잡한 모델을 시도하기 전에 **TF-IDF + 희소 행렬 + 선형 모델 또는 작은 MLP**로 기준 성능부터 확인한다.
- 평가지표를 먼저 확인하고, 그에 맞게 타깃을 변환하거나 손실함수를 고른다.
- 앙상블할 때 모델 종류뿐 아니라 **입력 표현을 다르게 하는 방법**도 고려한다.
- 실행 시간을 단계별로 기록해 두고, 시간이 많이 걸리는 단계부터 줄인다.